# Punto 1 · Notebook 00 – Descarga y filtrado de la estación MeteoNet

**Taller 1 – Aprendizaje Profundo** · Daniel Sebastian Velasco Munar

Este notebook se ejecuta **una sola vez** (idealmente en Google Colab). Hace tres cosas:

1. Descarga las observaciones de estaciones terrestres de MeteoNet para una zona (`NW` o `SE`) y los años 2016-2018 (~174 MB comprimidos por año).
2. Recorre los CSV por *chunks* y construye un **resumen de calidad por estación** (número de registros, % de valores faltantes por variable, cobertura temporal) → `results/punto1/resumen_estaciones_<zona>.csv`. Con esa tabla se justifica la selección de la estación.
3. Filtra la estación escogida, convierte unidades, remuestrea a frecuencia **horaria** y guarda `code/punto1_rnn_meteonet/data/estacion_<id>_horaria.csv` (archivo pequeño que sí se versiona en el repositorio). Los notebooks `01` y `02` parten de ese archivo.

Documentación de los datos: https://meteofrance.github.io/meteonet/english/data/ground-observations/

| Columna | Significado | Unidad original |
|---|---|---|
| `number_sta` | id de la estación | – |
| `lat`, `lon`, `height_sta` | ubicación y altitud | °, °, m |
| `date` | marca de tiempo (cada 6 min) | `YYYY-MM-DD HH:MM:SS` |
| `dd` | dirección del viento | ° |
| `ff` | velocidad del viento | m/s |
| `precip` | precipitación | kg/m² |
| `hu` | humedad relativa | % |
| `td` | punto de rocío | K |
| `t` | temperatura | K |
| `psl` | presión a nivel del mar | Pa |


## 0. Configuración

In [ ]:
# --- Parámetros del notebook -------------------------------------------------
ZONA = "NW"                      # "NW" (noroeste) o "SE" (sureste) de Francia
ANIOS = [2016, 2017, 2018]

# Estación a extraer. Dejar en None la primera vez: el notebook construye el resumen
# de estaciones y se escoge con base en él. Luego se fija aquí el id elegido.
ESTACION = None

# Chunk para leer los CSV grandes sin agotar la RAM de Colab (~12 GB)
CHUNKSIZE = 2_000_000

# Rutas
USAR_DRIVE = True                # cachear las descargas en Google Drive para no repetirlas
RUTA_DRIVE_CACHE = "/content/drive/MyDrive/meteonet_raw"


In [ ]:
import os, sys, tarfile, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

EN_COLAB = "google.colab" in sys.modules
print("Ejecutando en Colab:", EN_COLAB)


### 0.1 Repositorio y rutas de trabajo

En Colab se clona el repositorio para que las salidas queden en las carpetas correctas (`results/` y `code/punto1_rnn_meteonet/data/`) y se puedan hacer commit al final. En local se asume que el notebook se abre desde su carpeta dentro del repo.

In [ ]:
REPO_URL = "https://github.com/USUARIO/taller1-deep-learning.git"   # <-- ajustar

if EN_COLAB:
    if not Path("/content/taller1-deep-learning").exists():
        !git clone {REPO_URL} /content/taller1-deep-learning
    RAIZ_REPO = Path("/content/taller1-deep-learning")
else:
    # notebook ubicado en code/punto1_rnn_meteonet/
    RAIZ_REPO = Path.cwd().resolve().parents[1]

RUTA_DATA = RAIZ_REPO / "code" / "punto1_rnn_meteonet" / "data"
RUTA_RESULTS = RAIZ_REPO / "results" / "punto1"
RUTA_DATA.mkdir(parents=True, exist_ok=True)
RUTA_RESULTS.mkdir(parents=True, exist_ok=True)

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_RAW = Path(RUTA_DRIVE_CACHE)
else:
    RUTA_RAW = RAIZ_REPO / "code" / "punto1_rnn_meteonet" / "raw"   # ignorada por git
RUTA_RAW.mkdir(parents=True, exist_ok=True)

print("Repo:     ", RAIZ_REPO)
print("Crudos:   ", RUTA_RAW)
print("Salida:   ", RUTA_DATA)


## 1. Descarga y extracción de los archivos crudos

In [ ]:
BASE_URL = "https://meteonet.umr-cnrm.fr/dataset/data/{zona}/ground_stations/{zona}_ground_stations_{anio}.tar.gz"


def descargar(url: str, destino: Path, reintentos: int = 3) -> Path:
    """Descarga con barra de progreso; omite si el archivo ya existe."""
    if destino.exists() and destino.stat().st_size > 0:
        print(f"[ok] ya existe {destino.name} ({destino.stat().st_size/1e6:.0f} MB)")
        return destino
    for intento in range(1, reintentos + 1):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                total = int(r.headers.get("content-length", 0))
                tmp = destino.with_suffix(destino.suffix + ".part")
                with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=destino.name) as barra:
                    for bloque in r.iter_content(chunk_size=1 << 20):
                        f.write(bloque)
                        barra.update(len(bloque))
                tmp.rename(destino)
            return destino
        except Exception as e:
            print(f"  intento {intento} falló: {e}")
            time.sleep(5)
    raise RuntimeError(f"No se pudo descargar {url}")


def extraer_csv(tar_path: Path, carpeta: Path) -> Path:
    """Extrae el CSV del tar.gz (estructura: <ZONA><anio>.tar.gz -> <ZONA><anio>.csv). Omite si ya existe."""
    with tarfile.open(tar_path, "r:gz") as tar:
        miembros = [m for m in tar.getmembers() if m.name.lower().endswith(".csv")]
        assert miembros, f"El archivo {tar_path.name} no contiene CSV"
        m = miembros[0]
        destino = carpeta / Path(m.name).name
        if destino.exists():
            print(f"[ok] ya extraído {destino.name} ({destino.stat().st_size/1e9:.2f} GB)")
            return destino
        print(f"extrayendo {m.name} ...")
        m.name = Path(m.name).name      # aplanar rutas internas
        tar.extract(m, path=carpeta)
    return destino


CSV_POR_ANIO = {}
for anio in ANIOS:
    url = BASE_URL.format(zona=ZONA, anio=anio)
    tgz = descargar(url, RUTA_RAW / f"{ZONA}_ground_stations_{anio}.tar.gz")
    CSV_POR_ANIO[anio] = extraer_csv(tgz, RUTA_RAW)

CSV_POR_ANIO


In [ ]:
# Vistazo rápido a la estructura del primer archivo
muestra = pd.read_csv(CSV_POR_ANIO[ANIOS[0]], nrows=5)
display(muestra)
print(muestra.dtypes)


## 2. Resumen de calidad por estación

Se recorre cada CSV por *chunks* acumulando, por estación: número de registros, número de valores no nulos por variable, primera y última fecha, y metadatos (lat, lon, altura). Con eso se calcula el % de faltantes por variable y la **cobertura** (registros observados / registros esperados a 6 min en los 3 años ≈ 262 800).

Criterios sugeridos para escoger la estación (a justificar en el informe):

- Cobertura alta y presente en los **tres años** (serie larga y continua).
- Menor % de faltantes en `t` (variable objetivo) y en las covariables útiles (`hu`, `td`, `psl`, `ff`).
- Muchas estaciones **no reportan `psl`**; si se quiere usar presión como covariable hay que escogerla entre las que sí la tienen.


In [ ]:
VARIABLES = ["dd", "ff", "precip", "hu", "td", "t", "psl"]
COLUMNAS = ["number_sta", "lat", "lon", "height_sta", "date"] + VARIABLES

RUTA_RESUMEN = RUTA_RESULTS / f"resumen_estaciones_{ZONA}.csv"

if RUTA_RESUMEN.exists():
    resumen = pd.read_csv(RUTA_RESUMEN)
    print(f"[ok] resumen ya calculado: {RUTA_RESUMEN}")
else:
    acumulados = []
    for anio, ruta in CSV_POR_ANIO.items():
        t0 = time.time()
        for chunk in tqdm(pd.read_csv(ruta, usecols=COLUMNAS, chunksize=CHUNKSIZE, parse_dates=["date"]),
                          desc=f"{ZONA}{anio}"):
            g = chunk.groupby("number_sta")
            parcial = g.agg(n_registros=("date", "size"),
                            fecha_min=("date", "min"),
                            fecha_max=("date", "max"),
                            lat=("lat", "first"), lon=("lon", "first"), height_sta=("height_sta", "first"))
            for v in VARIABLES:
                parcial[f"nn_{v}"] = g[v].count()
            parcial["anio"] = anio
            acumulados.append(parcial.reset_index())
        print(f"  {anio}: {time.time()-t0:.0f} s")

    parciales = pd.concat(acumulados, ignore_index=True)

    # Consolidar por estación (sumas de conteos, min/max de fechas, años presentes)
    agg = {"n_registros": "sum", "fecha_min": "min", "fecha_max": "max",
           "lat": "first", "lon": "first", "height_sta": "first",
           "anio": pd.Series.nunique}
    agg.update({f"nn_{v}": "sum" for v in VARIABLES})
    resumen = parciales.groupby("number_sta").agg(agg).rename(columns={"anio": "n_anios"}).reset_index()

    registros_esperados = sum(pd.Timestamp(f"{a+1}-01-01") - pd.Timestamp(f"{a}-01-01") for a in ANIOS) / pd.Timedelta(minutes=6)
    resumen["cobertura"] = resumen["n_registros"] / registros_esperados
    for v in VARIABLES:
        resumen[f"pct_nan_{v}"] = 1 - resumen[f"nn_{v}"] / resumen["n_registros"]
    resumen = resumen.drop(columns=[f"nn_{v}" for v in VARIABLES])
    resumen = resumen.sort_values(["n_anios", "pct_nan_t", "cobertura"], ascending=[False, True, False])
    resumen.to_csv(RUTA_RESUMEN, index=False)
    print("guardado:", RUTA_RESUMEN)

print(f"{len(resumen)} estaciones en la zona {ZONA}")
resumen.head(20).style.format({"cobertura": "{:.1%}", **{f"pct_nan_{v}": "{:.1%}" for v in VARIABLES}})


In [ ]:
# Candidatas: 3 años completos, cobertura alta, pocos faltantes en temperatura y con presión disponible
candidatas = resumen.query("n_anios == @len(ANIOS) and cobertura > 0.95 and pct_nan_t < 0.02 and pct_nan_psl < 0.05")
print(f"{len(candidatas)} estaciones cumplen los criterios")
candidatas.head(15).style.format({"cobertura": "{:.1%}", **{f"pct_nan_{v}": "{:.1%}" for v in VARIABLES}})


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(resumen["lon"], resumen["lat"], c=resumen["pct_nan_t"], cmap="viridis_r", s=18, vmin=0, vmax=0.2)
if len(candidatas):
    ax.scatter(candidatas["lon"], candidatas["lat"], facecolors="none", edgecolors="red", s=60, label="candidatas")
    ax.legend()
plt.colorbar(sc, label="% faltantes en temperatura")
ax.set_xlabel("lon"); ax.set_ylabel("lat"); ax.set_title(f"Estaciones zona {ZONA} – calidad de la serie de temperatura")
fig.savefig(RUTA_RESULTS / f"mapa_estaciones_{ZONA}.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Extracción de la estación escogida

Fijar `ESTACION` con el id elegido (o dejar `None` para tomar automáticamente la primera candidata). Se hace una segunda pasada por los CSV conservando solo las filas de esa estación.

In [ ]:
if ESTACION is None:
    assert len(candidatas), "No hay candidatas con los criterios actuales; relajarlos o fijar ESTACION a mano."
    ESTACION = int(candidatas.iloc[0]["number_sta"])
print("Estación escogida:", ESTACION)
display(resumen.query("number_sta == @ESTACION").T)

partes = []
for anio, ruta in CSV_POR_ANIO.items():
    for chunk in tqdm(pd.read_csv(ruta, usecols=COLUMNAS, chunksize=CHUNKSIZE, parse_dates=["date"]),
                      desc=f"filtrando {ZONA}{anio}"):
        partes.append(chunk[chunk["number_sta"] == ESTACION])

serie_6min = (pd.concat(partes, ignore_index=True)
                .drop_duplicates(subset="date")
                .sort_values("date")
                .set_index("date"))
print(serie_6min.shape)
serie_6min.head()


## 4. Conversión de unidades y remuestreo horario

- `t`, `td`: K → °C. `psl`: Pa → hPa.
- Remuestreo a 1 h: promedio para las variables de estado; **suma** para precipitación; la dirección del viento se promedia vectorialmente (`dd`, `ff` → componentes `u`, `v`).
- Se reindexa a una malla horaria completa para que las horas sin observación queden como `NaN` explícitos (la imputación se decide y documenta en el notebook `01`, sin usar información futura).


In [ ]:
df = serie_6min.copy()
df["t_c"] = df["t"] - 273.15
df["td_c"] = df["td"] - 273.15
df["psl_hpa"] = df["psl"] / 100.0

# viento a componentes (convención meteorológica: dd es la dirección DESDE la que sopla)
rad = np.deg2rad(df["dd"])
df["u"] = -df["ff"] * np.sin(rad)
df["v"] = -df["ff"] * np.cos(rad)

horaria = pd.DataFrame({
    "t_c":      df["t_c"].resample("1h").mean(),
    "td_c":     df["td_c"].resample("1h").mean(),
    "hu":       df["hu"].resample("1h").mean(),
    "psl_hpa":  df["psl_hpa"].resample("1h").mean(),
    "ff":       df["ff"].resample("1h").mean(),
    "u":        df["u"].resample("1h").mean(),
    "v":        df["v"].resample("1h").mean(),
    "precip":   df["precip"].resample("1h").sum(min_count=1),
    "n_obs":    df["t"].resample("1h").count(),   # cuántas de las 10 observaciones posibles hubo
})

indice_completo = pd.date_range(f"{ANIOS[0]}-01-01 00:00", f"{ANIOS[-1]}-12-31 23:00", freq="1h")
horaria = horaria.reindex(indice_completo)
horaria.index.name = "date"

print(horaria.shape)
print("faltantes por columna:")
print(horaria.isna().mean().round(4))
horaria.describe().T


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
horaria["t_c"].plot(ax=axes[0], lw=0.5); axes[0].set_ylabel("Temperatura (°C)")
horaria["n_obs"].plot(ax=axes[1], lw=0.5); axes[1].set_ylabel("obs / hora")
axes[0].set_title(f"Estación {ESTACION} – serie horaria {ANIOS[0]}–{ANIOS[-1]}")
fig.savefig(RUTA_RESULTS / f"serie_horaria_estacion_{ESTACION}.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Guardar el archivo que usan los siguientes notebooks

In [ ]:
RUTA_SALIDA = RUTA_DATA / f"estacion_{ESTACION}_horaria.csv"
horaria.round(3).to_csv(RUTA_SALIDA)

meta = resumen.query("number_sta == @ESTACION").iloc[0]
metadatos = {
    "zona": ZONA, "estacion": ESTACION, "anios": ANIOS,
    "lat": float(meta["lat"]), "lon": float(meta["lon"]), "height_sta": float(meta["height_sta"]),
    "frecuencia": "1h", "n_filas": int(len(horaria)),
    "pct_nan_t_c": float(horaria["t_c"].isna().mean()),
    "columnas": list(horaria.columns),
}
with open(RUTA_DATA / f"estacion_{ESTACION}_metadatos.json", "w") as f:
    json.dump(metadatos, f, indent=2)

print(f"guardado {RUTA_SALIDA} ({RUTA_SALIDA.stat().st_size/1e6:.2f} MB)")
metadatos


## 6. Guardar los resultados en el repositorio (solo en Colab)

Los archivos crudos quedan en Drive (cache) y están ignorados por git. Solo se versionan el CSV horario, los metadatos, el resumen de estaciones y las figuras.

In [ ]:
if EN_COLAB:
    %cd {RAIZ_REPO}
    !git config user.name  "Daniel Sebastian Velasco Munar"
    !git config user.email "ds.velasco2170@gmail.com"
    !git status --short
    # Descomentar para hacer commit y push (requiere token de GitHub configurado en la URL o en git credential)
    # !git add results/punto1 code/punto1_rnn_meteonet/data
    # !git commit -m "Punto 1: resumen de estaciones y serie horaria de la estación {ESTACION}"
    # !git push
